# Exploratory Data Analysis & Feature Decision Rationale
## Diabetes 130-US Hospitals (1999-2008) Readmission Prediction Pipeline
This notebook details the statistical properties, missingness structure, class balance, clinical subgroup disparity rates, and feature selection rationales for the hospital readmission model.

In [ ]:
import pandas as pd
import numpy as np
import json

# Load raw dataset
df = pd.read_csv('../data/raw/diabetic_data.csv')
print(f'Raw dataset shape: {df.shape}')
df.head()

### 1. Target Definition & CMS HRRP Clinical Context
The readmission outcome contains three levels: `<30` (early readmission within 30 days), `>30` (late readmission after 30 days), and `NO` (no recorded readmission).
Under the CMS Hospital Readmissions Reduction Program (HRRP), federal hospital reimbursement penalties apply strictly to uncoordinated 30-day readmissions. Therefore, the target is defined as:
- **Positive Class (1):** `readmitted == '<30'`
- **Negative Class (0):** `readmitted in ['>30', 'NO']`

In [ ]:
# Target distribution
print('Raw 3-class distribution:')
print(df['readmitted'].value_counts(normalize=True) * 100)

df['target'] = (df['readmitted'] == '<30').astype(int)
print('\nBinary 30-day early readmission rate:')
print(df['target'].value_counts(normalize=True) * 100)

### 2. Missing Values Audit & Handling Policy
In this dataset, missing values are denoted by `'?'`.
- **Weight (>96% missing):** Dropped due to extreme sparsity across facilities.
- **Payer code (39.6% missing) & Medical specialty (49.1% missing):** Administrative/billing variables excluded to prevent overfitting.
- **Race (2.2% missing):** Imputed to explicit category `'Other/Unknown'` to support algorithmic fairness audits.
- **Diagnoses (0.02% - 1.4% missing):** Primary ICD-9 diagnosis grouped into standard clinical categories.

In [ ]:
missing = ((df == '?') | (df.isna())).sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Percentage': missing_pct})
missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Percentage', ascending=False)

### 3. Patient Duplication & Train-Test Leakage Integrity
Many patients had multiple hospital admissions over the 10-year tracking period. To prevent data leakage, all encounters for a given patient must be strictly partitioned together using `StratifiedGroupKFold` on `patient_nbr`.

In [ ]:
enc_counts = df.groupby('patient_nbr')['encounter_id'].count()
print(f'Total encounters: {len(df)}')
print(f'Unique patients: {df.patient_nbr.nunique()}')
print(f'Patients with multiple encounters: {(enc_counts > 1).sum()} ({(enc_counts > 1).mean()*100:.1f}%)')
print(f'Max encounters for a single patient: {enc_counts.max()}')

### 4. Terminal Disposition Exclusions
Patients discharged to hospice or who expired during the inpatient stay cannot experience 30-day readmissions. Including them artificially inflates negative survival predictions.

In [ ]:
dead_ids = {11, 13, 14, 19, 20, 21, '11', '13', '14', '19', '20', '21'}
dead_mask = df['discharge_disposition_id'].astype(str).isin(dead_ids)
print(f'Terminal/hospice encounters excluded: {dead_mask.sum()}')
df_clean = df[~dead_mask].copy().reset_index(drop=True)
df_clean['target'] = (df_clean['readmitted'] == '<30').astype(int)
print(f'Remaining clean encounters: {len(df_clean)}')
print(f'Clean 30-day readmission rate: {df_clean.target.mean()*100:.2f}%')

### 5. Empirical Readmission Disparities across Clinical & Demographic Cohorts

In [ ]:
print('--- Readmission Rate by Prior Inpatient Hospitalizations ---')
print(df_clean.groupby(pd.cut(df_clean['number_inpatient'], [-1, 0, 1, 2, 5, 50]))['target'].agg(['count', 'mean']))

print('\n--- Readmission Rate by Age Cohort ---')
print(df_clean.groupby('age')['target'].agg(['count', 'mean']))

print('\n--- Readmission Rate by Race ---')
print(df_clean.groupby(df_clean['race'].replace('?', 'Other/Unknown'))['target'].agg(['count', 'mean']))